## 1. Multi-Head Attention (다중 헤드 어텐션)
- **왜 하나가 아닌 여러 개(Multi)를 쓸까요?** : 사람도 문장을 분석할 때 "문법 구조(주어와 동사의 관계)"를 주로 보는 시각과 "감성(긍정/부정 뉘앙스)"을 주로 보는 시각이 다릅니다. 단일 어텐션만 쓰면 텐서가 한 가지 관점에만 편향될 수 있습니다.
- **작동 원리** : 앞서 3교시에서 배운 Q, K, V 벡터를 여러 개의 머리(Head, 보통 8개나 12개)로 쪼개어 각각 독립적으로 Self-Attention 연산을 병렬 수행합니다. 이렇게 각기 다른 관점으로 파악한 문맥 텐서들을 마지막에 다시 하나로 이어 붙여(Concat), 훨씬 다채롭고 입체적인 문맥을 포착합니다.

## 2. Positional Encoding (위치 인코딩)
- **왜 필요할까요?** : 트랜스포머는 RNN을 버리고 오직 행렬 곱셈을 사용하므로, 문장의 모든 단어가 '동시에' 입력됩니다. 즉, 모델 입장에서는 '순서'라는 개념 자체가 존재하지 않아서 "나 는 밥 을 먹는다"와 "밥 은 나 를 먹는다"를 수학적으로 완전히 동일하게 받아들이는 치명적 문제가 생깁니다.
- **해결책** : 이를 해결하기 위해 입력 단어 임베딩 벡터에 "나는 1번째 위치다", "나는 2번째 위치다"라는 고유의 식별 정보(사인/코사인 파동 함수 등)를 덧셈 연산(+)으로 주입합니다. 이 덕분에 트랜스포머는 병렬 연산의 속도를 유지하면서도 단어의 순서를 완벽히 파악할 수 있습니다.

## 3. Layer Normalization과 Residual Connection
- **Residual Connection (잔차 연결)** : 블록을 통과한 출력값에 통과하기 전의 원래 입력값을 그대로 더해주는(`x + F(x)`) 기법입니다. 신경망 레이어가 깊어질수록 최초의 정보가 희미해지는 현상과 역전파 시 미분값이 사라지는(Gradient Vanishing) 현상을 막아주는 고속도로 역할을 합니다.
- **Layer Normalization (층 정규화)** : 각 데이터 샘플의 차원들을 평균 0, 분산 1로 정규화합니다. 텐서 내부의 값이 너무 커지거나 작아지는 것을 막아주어, 거대한 트랜스포머 모델이 안정적이고 빠르게 학습될 수 있도록 돕습니다.

## 4. 오리지널 Transformer의 파이프라인 구조 요약
기계 번역을 위해 탄생한 오리지널 트랜스포머는 크게 두 파트로 나뉩니다.
- **Encoder (인코더)**: 번역할 원문을 입력받아 Self-Attention으로 철저히 분석하여, 문맥이 완벽하게 압축된 하나의 벡터 덩어리로 만듭니다. (이 구조만 떼어내 발전시킨 것이 **BERT** 입니다.)
- **Decoder (디코더)**: 인코더가 압축해준 문맥 정보를 넘겨받아(Cross-Attention), 앞서 생성한 단어들을 참고하며 번역된 단어를 순차적으로 하나씩 만들어냅니다. (이 구조만 떼어내 발전시킨 것이 **GPT** 입니다.)

In [2]:
%pip install transformers 

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 10.6/10.6 MB 83.0 MB/s  0:00:00
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 663.6/663.6 kB ?  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 119.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 77.7 MB/s  0:00:00

   ------ ---------------------------------  2/12 [hf-xet]
   ---------------- -----------------------  5/12 [httpcore]
   -------------------- -------------------  6/12 [anyio]
   -------------------- -------------------  6/12 [anyio]
   -------------------------- -------------  8/12 [httpx]
   ------------------------------ ---------  9/12 [huggingface-hub]
   ------------------------------ ---------  9/12 [huggingface-

In [9]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# 리턴값 분석을 위한 출력 -SDPA 고속연산은 어텐션 가중치를 외부로 빼지 않음
model = BertModel.from_pretrained("bert-base-uncased", attn_implementation='eager')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
text = "The cat sat on the mat."
inputs = tokenizer(text, return_tensors='pt')
print(inputs)
# 모델 통과 및 어텐션 텐서 변환
outputs = model(**inputs, output_attentions=True)
# # 토큰화된 단어들 확인
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f'추출된 토큰들 : {tokens}')

all_attentions = outputs.attentions
print(f'모델이 반환한 전체 레이어 개수 : {len(all_attentions)}')
print(f'첫 번째 레이어의 어텐션 텐서 크기 : {all_attentions[0].shape}')

{'input_ids': tensor([[  101,  1996,  4937,  2938,  2006,  1996, 13523,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}
추출된 토큰들 : ['[CLS]', 'the', 'cat', 'sat', 'on', 'the', 'mat', '.', '[SEP]']
모델이 반환한 전체 레이어 개수 : 12
첫 번째 레이어의 어텐션 텐서 크기 : torch.Size([1, 12, 9, 9])


In [ ]:
# 첫 번째 레이어의 어텐션 텐서 크기 : torch.Size([1, 12, 9, 9])
